In [0]:
"""
AeroPulse Enterprise Lakehouse Platform.

Reusable Bronze ingestion utilities.

This module contains common functionality for reading
source deliveries, adding Bronze metadata, and writing
to Delta Bronze tables.
"""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def read_source_delivery(
    spark,
    source_delivery_path: str,
    file_format: str,
) -> DataFrame:
    """
    Read a source delivery using the specified file format.

    Parameters
    ----------
    spark:
        Active SparkSession.
    source_delivery_path:
        Path of the source delivery.
    file_format:
        Source file format.

    Returns
    -------
    DataFrame
        Source delivery DataFrame.
    """

    normalized_file_format = file_format.lower()

    if normalized_file_format == "csv":

        return (
            spark.read
            .option("header", "true")
            .csv(source_delivery_path)
        )

    elif normalized_file_format == "json":

        return (
            spark.read
            .json(source_delivery_path)
        )

    elif normalized_file_format == "parquet":

        return (
            spark.read
            .parquet(source_delivery_path)
        )

    else:

        raise ValueError(
            f"Unsupported file format: {file_format}"
        )


def add_bronze_metadata(
    source_dataframe: DataFrame,
    source_system: str,
    pipeline_run_id: str,
) -> DataFrame:
    """
    Add AeroPulse Bronze technical metadata columns.
    """

    return (
        source_dataframe
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file_path",
            F.input_file_name()
        )
        .withColumn(
            "_source_system",
            F.lit(source_system)
        )
        .withColumn(
            "_pipeline_run_id",
            F.lit(pipeline_run_id)
        )
    )


def write_to_bronze(
    bronze_dataframe: DataFrame,
    bronze_table: str,
) -> None:
    """
    Write processed data to a Bronze Delta table.
    """

    (
        bronze_dataframe.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )

In [0]:
%run "./delivery_discovery.py"

In [0]:
%run "./ingestion_registry.py"

In [0]:
def ingest_bronze_deliveries(
    spark,
    dbutils,
    source_path: str,
    source_entity: str,
    source_system: str,
    file_format: str,
    bronze_table: str,
    registry_table: str,
    pipeline_run_id: str,
):
    """
    Execute reusable delivery-level Bronze ingestion.

    This function orchestrates:

    1. Source delivery discovery
    2. Idempotency validation
    3. Delivery registration
    4. Source reading
    5. Bronze metadata enrichment
    6. Bronze Delta writing
    7. Delivery status management

    Returns
    -------
    dict
        Pipeline ingestion metrics.
    """

    delivery_paths = discover_source_deliveries(
        dbutils=dbutils,
        source_path=source_path,
        source_entity=source_entity,
    )

    records_read_total = 0
    records_inserted_total = 0
    deliveries_processed = 0
    deliveries_skipped = 0

    for delivery_path in delivery_paths:

        print(
            f"Checking delivery: {delivery_path}"
        )

        if is_delivery_processed(
            spark=spark,
            registry_table=registry_table,
            source_delivery_path=delivery_path,
        ):

            deliveries_skipped += 1

            print(
                "Already successfully processed. "
                "Skipping."
            )

            continue

        delivery_name = (
            delivery_path
            .rstrip("/")
            .split("/")[-1]
        )

        register_delivery_start(
            spark=spark,
            registry_table=registry_table,
            source_delivery_path=delivery_path,
            source_file_name=delivery_name,
            source_system=source_system,
            source_entity=source_entity,
            file_format=file_format.upper(),
            pipeline_run_id=pipeline_run_id,
        )

        try:

            source_df = read_source_delivery(
                spark=spark,
                source_delivery_path=delivery_path,
                file_format=file_format,
            )

            records_read = source_df.count()

            bronze_df = add_bronze_metadata(
                source_dataframe=source_df,
                source_system=source_system,
                pipeline_run_id=pipeline_run_id,
            )

            write_to_bronze(
                bronze_dataframe=bronze_df,
                bronze_table=bronze_table,
            )

            records_inserted = bronze_df.count()

            mark_delivery_success(
                spark=spark,
                registry_table=registry_table,
                source_delivery_path=delivery_path,
                pipeline_run_id=pipeline_run_id,
                records_read=records_read,
                records_inserted=records_inserted,
            )

            records_read_total += records_read
            records_inserted_total += records_inserted
            deliveries_processed += 1

            print(
                f"Successfully processed: "
                f"{delivery_name}"
            )

        except Exception as error:

            mark_delivery_failed(
                spark=spark,
                registry_table=registry_table,
                source_delivery_path=delivery_path,
                pipeline_run_id=pipeline_run_id,
                error_message=str(error),
            )

            raise

    return {
        "deliveries_discovered": len(delivery_paths),
        "deliveries_processed": deliveries_processed,
        "deliveries_skipped": deliveries_skipped,
        "records_read": records_read_total,
        "records_inserted": records_inserted_total,
    }